# Generate Label Studio XML from Additional Needs Taxonomy

In [22]:
import pandas as pd
import ast
from xml.sax.saxutils import escape

In [23]:
with open('label-studio-ui-template.xml', 'r') as f:
    template = f.read().splitlines() 

In [24]:
taxonomy = pd.read_csv('data/output/taxonomy_v2_autogen.csv')
taxonomy.head()

,top_level_category,high_level_category,category_description,values_hint,high_level_cat_label,cat_label,regex
0,Disability,Disability,Sensory,"['Deaf', 'Hearing impairment', 'Visually impai...",cat_disability_disability,cat_disability_disability_sensory,\bdeaf\b|\bBSL\b|\bhearing (?:loss|impairment|...
1,Disability,Adaptation,Requires adapted property,"['Minor adaptations', 'Major adaptation']",cat_disability_adaptation,cat_disability_adaptation_requires_adapted_pro...,\bstairlift\b|\bwet room\b|\blevel access\b|\b...
2,Safety & Risk,Safety & Risk,Domestic abuse,"['Alleged Perpetrator', 'Survivor']",cat_safety_risk_safety_risk,cat_safety_risk_safety_risk_domestic_abuse,\bdomestic abuse\b|\bdomestic violence\b|\bDV\...
3,Safety & Risk,Safety & Risk,Fire-related risks,"['Arson', 'Fire hazards in the house']",cat_safety_risk_safety_risk,cat_safety_risk_safety_risk_firerelated_risks,\barson\b|\bfire hazard\b|\bfire risk\b|\bfire...
4,Safety & Risk,Safety & Risk,Risk of exploitation,"['Financial', 'Sexual', 'Criminal', 'Cuckooing...",cat_safety_risk_safety_risk,cat_safety_risk_safety_risk_risk_of_exploitation,\bcuckooing\b|\bfinancial (?:exploitation|abus...


In [25]:
# Colour per high_level_cat_label
GROUP_COLOURS = {
    'Adaptation':            '#5DADE2',  # Light Blue (Distinct from Housing)
    'Disability':            '#58D68D',  # Soft Green (Distinct from Health/Mobility)
    'Safety & Risk':         '#EC7063',  # Soft Red (High visibility for risk)
    'Communication needs':   '#F4D03F',  # Bright Yellow/Gold (High contrast)
    'Addiction':             '#CD6155',  # Muted Red/Terra Cotta (Distinct from Safety)
    'Care':                  '#48C9B0',  # Turquoise (Distinct from Health/Adaptation)
    'Financial':             '#F5B041',  # Orange-Gold (Distinct from Communication)
    'Health':                '#7DCEA0',  # Pale Green (Distinct from Disability/Mobility)
    'Housing Conditions':    '#85C1E9',  # Sky Blue (Distinct from Adaptation)
    'Life Events':           '#EB984E',  # Soft Orange (Distinct from Financial)
    'Mobility':              '#7FB3D5',  # Steel Blue (Distinct from Adaptation/Housing)
}

DEFAULT_COLOUR = '#95a5a6'

In [26]:
category_labels_xml = []

# Generate labels selection coloured by high_level_category
for high_level, group in taxonomy.groupby('high_level_category'):
    category_labels_xml.append('')
    colour = GROUP_COLOURS.get(str(high_level), DEFAULT_COLOUR)
    for row in group.itertuples():
        hint = ", ".join(ast.literal_eval(str(row.values_hint)))
        category_labels_xml.append(
            f'          <Label value="{row.cat_label}" html="{escape(str(row.category_description))}" background="{colour}" hint="{hint}"/>'
        )

category_labels_xml[:4]

['',
 '          <Label value="cat_disability_adaptation_requires_adapted_property" html="Requires adapted property" background="#5DADE2" hint="Minor adaptations, Major adaptation"/>',
 '',
 '          <Label value="cat_vulnerability_addiction_gambling" html="Gambling" background="#CD6155" hint="Y, N"/>']

In [27]:
relation = f'''      <Relation 
        value="AFFECTS" 
        fromName="need_labels" 
        toName="entity_labels" 
        label="{",".join(taxonomy['cat_label'].to_list())}"
      />'''

In [28]:
i = template.index("      <AN_PLACEHOLDER>")

new_file = template[:i] + category_labels_xml + template[i+1:]

relation_i = new_file.index("      <RELATION_PLACEHOLDER>")
new_file[relation_i] = relation

with open('data/output/label-studio-ui.xml', 'w') as f:
    f.write('\n'.join(new_file))